# Wildfire Ignition Prediction — Data Cleaning & Feature Engineering

Cleaned, consolidated pipeline. This version fixes two issues found in the working/debugging
version of this notebook:

1. **Sort order was implicit, not enforced.** `seq_id` window-chunking relies on rows being
   sorted by `(latitude, longitude, datetime)`. The working version relied on the raw CSV's
   native row order already matching this — verified empirically (0 date gaps, clean 75-row
   divisibility) but never enforced. This version sorts explicitly.
2. **`is_holiday` bug.** Comparing `event_date` (datetime64) directly against a `holidays.US`
   object via `.isin()` silently returns all-False. Fixed by comparing `.dt.date` instead.


In [ ]:
import pandas as pd
import numpy as np


## 1. Load & Sentinel Cleaning

GRIDMET encodes missing values as the sentinel `32767`, not `NaN`. This must be masked to `NaN` explicitly — `dropna()` alone does nothing, since `32767` is a valid-looking number, not a null.

In [ ]:
RAW_FEATURES = ["pr","rmax","rmin","sph","srad","tmmn","tmmx",
                "vs","bi","fm100","fm1000","erc","etr","pet","vpd"]

df_raw = pd.read_csv('Wildfire_Dataset.csv', parse_dates=['datetime'])

# Mask sentinel values as NaN. Do NOT call dropna() here — dropping rows before
# window (seq_id) assignment desyncs the 75-row block structure.
df_raw[RAW_FEATURES] = df_raw[RAW_FEATURES].mask(df_raw[RAW_FEATURES] >= 30000)

print(df_raw[RAW_FEATURES].isna().sum())
print(df_raw[RAW_FEATURES].describe())


## 2. Sort & Window Reconstruction

Each window is a fixed 75-row block (60 pre-event days + event day + 15 post-event days), following the FireCastRL dataset's construction. `seq_id` chunking via `row_number // 75` is only valid on data sorted by `(latitude, longitude, datetime)` — sorting explicitly here, rather than relying on the CSV's native row order.

In [ ]:
df_raw = df_raw.sort_values(['latitude', 'longitude', 'datetime']).reset_index(drop=True)

print(df_raw.shape)
print(df_raw.shape[0] / 75)  # must be a clean integer

df_raw['seq_id'] = np.arange(len(df_raw)) // 75


### 2.1 Remove Contradictory-Label Windows

Some (lat, lon, date) triplets appear twice in the raw data with **conflicting** `Wildfire` labels — a real construction bug in the source dataset (two independently sampled windows landing on identical coordinates and overlapping dates). These whole windows are dropped, not individual rows, to preserve the 75-row block structure.

In [ ]:
dupe_keys = df_raw[df_raw.duplicated(subset=['latitude','longitude','datetime'], keep=False)]
conflicting = dupe_keys.groupby(['latitude','longitude','datetime']).filter(lambda g: g['Wildfire'].nunique() > 1)

bad_seq_ids = conflicting['seq_id'].unique()
print("Conflicting rows:", len(conflicting))
print("Windows affected:", len(bad_seq_ids))

df_raw_clean = df_raw[~df_raw['seq_id'].isin(bad_seq_ids)].reset_index(drop=True)
print(df_raw_clean.shape)


### 2.2 Remove Exact-Duplicate-Row Windows

Separately, some windows contain an exact duplicate row (same lat/lon/date/weather/label), which silently causes one day to be double-counted and another day dropped from that window. These windows are also dropped entirely.

In [ ]:
exact_dupes_in_kept = df_raw_clean[df_raw_clean.duplicated()]
print("Exact duplicate rows found:", len(exact_dupes_in_kept))
print("Windows affected:", exact_dupes_in_kept['seq_id'].nunique())

bad_seq_ids_2 = exact_dupes_in_kept['seq_id'].unique()
df_final = df_raw_clean[~df_raw_clean['seq_id'].isin(bad_seq_ids_2)].reset_index(drop=True)

print(df_final.shape)
print(df_final.shape[0] / 75)  # should be a clean integer again


In [ ]:
# rebuild seq_id fresh — the previous one has gaps from the dropped windows
df_final = df_final.drop(columns=['seq_id'])
df_final['seq_id'] = np.arange(len(df_final)) // 75

window_labels = df_final.groupby('seq_id')['Wildfire'].apply(lambda x: (x == 'Yes').any()).astype(int)

print("Total windows:", len(window_labels))
print(window_labels.value_counts(normalize=True))


### 2.3 Verify Window Integrity

Confirm no window has internal date gaps (would indicate a chunking desync).

In [ ]:
def has_gap(g):
    dates = g['datetime'].sort_values()
    return (dates.diff().dt.days.dropna() != 1).any()

gap_check = df_final.groupby('seq_id').apply(has_gap)
print("Windows with date gaps:", gap_check.sum(), "out of", len(gap_check))


## 3. Unit Conversion & Renaming

`tmmn`/`tmmx` are in Kelvin despite some source documentation claiming °C — verified against raw value ranges (~260-300). Converting and renaming all columns to descriptive names.

In [ ]:
df_final['tmmn'] = df_final['tmmn'] - 273.15
df_final['tmmx'] = df_final['tmmx'] - 273.15

rename_map = {
    'pr': 'precip_mm', 'rmax': 'rh_max_pct', 'rmin': 'rh_min_pct',
    'sph': 'spec_humidity_kgkg', 'srad': 'solar_rad_wm2',
    'tmmn': 'temp_min_c', 'tmmx': 'temp_max_c', 'vs': 'wind_speed_ms',
    'bi': 'burning_index', 'fm100': 'fuel_moist_100hr_pct',
    'fm1000': 'fuel_moist_1000hr_pct', 'erc': 'energy_release_component',
    'etr': 'evapotrans_mm', 'pet': 'pot_evapotrans_mm', 'vpd': 'vpd_kpa',
}
df_final = df_final.rename(columns=rename_map)

FEATURES = list(rename_map.values())


## 4. Handle Remaining NaN Windows

Sentinel-masked NaNs may still be present in the pre-event (first 60 days) period of some windows. Checking scope and outcome-class distribution before deciding whether to drop.

In [ ]:
n_windows = len(df_final) // 75
arr = df_final[FEATURES].values.reshape(n_windows, 75, len(FEATURES))
pre = arr[:, :60, :]

windows_with_nan = np.isnan(pre).any(axis=(1, 2))
print("Windows with NaN in pre-event period:", windows_with_nan.sum(), "/", n_windows)

window_labels = df_final.groupby('seq_id')['Wildfire'].apply(lambda x: (x == 'Yes').any()).astype(int).values
print("Positive rate in NaN-affected windows:", window_labels[windows_with_nan].mean())
print("Positive rate in clean windows:", window_labels[~windows_with_nan].mean())


**Finding:** NaN-affected windows are exclusively negative-class (0% fire rate). Plausible explanation: remote "far-negative" sample locations (≥100km from any real fire) are more likely to fall in areas with GRIDMET data gaps than real, well-documented IRWIN fire locations. Safe to drop — zero fire-positive loss.

In [ ]:
bad_nan_seq_ids = df_final.groupby('seq_id').filter(
    lambda g: g[FEATURES].iloc[:60].isna().any().any()
)['seq_id'].unique()

df_final_clean = df_final[~df_final['seq_id'].isin(bad_nan_seq_ids)].reset_index(drop=True)
print(df_final_clean.shape, df_final_clean.shape[0] / 75)


## 5. Feature Engineering: Window Collapse

Each window's **pre-event 60 days only** (excluding the ignition day and 15-day tail, which would leak in-fire weather) is collapsed into mean/max/min/std/slope per weather variable — turning a 75-day sequence problem into a standard tabular ML problem.

In [ ]:
df_final = df_final_clean.reset_index(drop=True)

PRE_EVENT_DAYS = 60
n_windows = len(df_final) // 75

arr = df_final[FEATURES].values.reshape(n_windows, 75, len(FEATURES))
pre = arr[:, :PRE_EVENT_DAYS, :]

mean = pre.mean(axis=1)
mx = pre.max(axis=1)
mn = pre.min(axis=1)
std = pre.std(axis=1)

x = np.arange(PRE_EVENT_DAYS)
x_centered = x - x.mean()
denom = (x_centered ** 2).sum()
slope = ((pre - pre.mean(axis=1, keepdims=True)) * x_centered[None, :, None]).sum(axis=1) / denom

X = np.concatenate([mean, mx, mn, std, slope], axis=1)

feature_names = (
    [f'{c}_mean' for c in FEATURES] +
    [f'{c}_max' for c in FEATURES] +
    [f'{c}_min' for c in FEATURES] +
    [f'{c}_std' for c in FEATURES] +
    [f'{c}_slope' for c in FEATURES]
)

feature_df = pd.DataFrame(X, columns=feature_names)

wildfire_arr = df_final['Wildfire'].to_numpy(dtype=str).reshape(n_windows, 75)
feature_df['label'] = (wildfire_arr == 'Yes').any(axis=1).astype(int)
feature_df['latitude'] = df_final['latitude'].values[::75]
feature_df['longitude'] = df_final['longitude'].values[::75]
feature_df['event_date'] = df_final['datetime'].values[PRE_EVENT_DAYS::75]

print(feature_df.shape)
print(feature_df['label'].value_counts(normalize=True))


### 5.1 Verify No Cross-Window Time Overlap

Since the same (lat, lon) can be reused across different-year windows, confirm no two windows at the same location have overlapping pre-event periods — which would make a simple random train/test split unsafe.

In [ ]:
feature_df_sorted = feature_df.sort_values(['latitude', 'longitude', 'event_date'])
gap_days = feature_df_sorted.groupby(['latitude', 'longitude'])['event_date'].diff().dt.days

print(gap_days.describe())
print("Windows with <75-day gap to a same-location neighbor:", (gap_days < 75).sum())


**Result:** minimum gap is 76 days — no overlap. A plain random train/test split is safe.

## 6. Additional Features: Holidays & Weekends

Human-ignition-relevant temporal features. Note: comparing datetime64 values directly against a `holidays.US` object via `.isin()` silently returns all-False — must compare against `.dt.date` instead.

In [ ]:
import holidays

us_holidays = holidays.US(years=range(2013, 2026))

feature_df['is_weekend'] = feature_df['event_date'].dt.dayofweek.isin([5, 6]).astype(int)
feature_df['is_holiday'] = feature_df['event_date'].dt.date.isin(us_holidays).astype(int)
feature_df['is_july4_week'] = (
    (feature_df['event_date'].dt.month == 7) &
    (feature_df['event_date'].dt.day.between(1, 7))
).astype(int)

print(feature_df[['is_weekend', 'is_holiday', 'is_july4_week']].mean())
# is_weekend ~0.286 (2/7), is_holiday ~0.027 (~10/365) confirm correct behavior


## 7. Population Density (NASA SEDAC GPWv4)

Linear interpolation between the 2010/2015/2020 snapshots for events within that range; linear extrapolation using each point's own 2015→2020 growth slope beyond 2020. Raster nodata sentinel (`-3.4028230607370965e+38`, float32 min) is masked to NaN first.

In [ ]:
import rasterio

years = [2010, 2015, 2020]
NODATA = -3.4028230607370965e+38
coords = list(zip(feature_df['longitude'], feature_df['latitude']))
event_years = feature_df['event_date'].dt.year.values

pop_by_year = {}
for yr in years:
    with rasterio.open(f'gpw_v4_population_density_rev11_{yr}_30_sec.tif') as src:
        vals = np.array([v[0] for v in src.sample(coords)])
        pop_by_year[yr] = np.where(vals <= NODATA * 0.9, np.nan, vals)

pop_matrix = np.stack([pop_by_year[y] for y in years], axis=1)

def interpolate_extrapolate(pop_row, event_year):
    if event_year <= 2020:
        return np.interp(event_year, years, pop_row)
    else:
        slope = (pop_row[2] - pop_row[1]) / (2020 - 2015)
        return max(pop_row[2] + slope * (event_year - 2020), 0)

feature_df['pop_density'] = [
    interpolate_extrapolate(pop_matrix[i], event_years[i])
    for i in range(len(feature_df))
]

print(feature_df['pop_density'].describe())
print("Remaining NaN (dropped below):", feature_df['pop_density'].isna().sum())


In [ ]:
feature_df = feature_df.dropna(subset=['pop_density']).reset_index(drop=True)
print(feature_df.shape)


## 8. Land Cover (USGS NLCD)

**Important:** NLCD is distributed in Albers Conic Equal Area projection (EPSG:5070, meters), not lat/lon. Coordinates must be reprojected before sampling — feeding raw lat/lon directly returns a uniform nodata/fill code (250) for every point.

In [ ]:
from pyproj import Transformer

with rasterio.open('Annual_NLCD_LndCov_2021_CU_C1V2.tif') as src:
    transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
    x_proj, y_proj = transformer.transform(feature_df['longitude'].values, feature_df['latitude'].values)
    coords_proj = list(zip(x_proj, y_proj))
    land_cover_codes = [v[0] for v in src.sample(coords_proj)]

feature_df['land_cover_code'] = land_cover_codes
print(feature_df['land_cover_code'].value_counts())


In [ ]:
feature_df = feature_df[feature_df['land_cover_code'] != 250].reset_index(drop=True)

def group_landcover(code):
    if code in [21, 22, 23, 24]: return 'developed'
    elif code in [41, 42, 43]: return 'forest'
    elif code in [51, 52]: return 'shrubland'
    elif code in [71, 72, 73, 74]: return 'grassland'
    elif code in [81, 82]: return 'agriculture'
    elif code in [90, 95]: return 'wetland'
    elif code in [11, 12]: return 'water'
    elif code in [31]: return 'barren'
    else: return 'other'

feature_df['land_cover_group'] = feature_df['land_cover_code'].apply(group_landcover)
feature_df = pd.get_dummies(feature_df, columns=['land_cover_group'], prefix='lc')

print(feature_df.shape)
print(feature_df.filter(like='lc_').sum())


## 9. Correlation Sanity Check

Confirms the feature set is free of the earlier sentinel-value corruption, which had produced spurious ~0.999 correlations across physically unrelated variables. This clean run should show only physically explainable relationships (e.g. actual vs. potential evapotranspiration, fuel moisture across timelags).

In [ ]:
X_check = feature_df.drop(columns=['label', 'latitude', 'longitude', 'event_date', 'land_cover_code'])
corr_matrix = X_check.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

high_corr = [col for col in upper.columns if any(upper[col] > 0.9)]
print("Features with a correlation > 0.9 to another feature:", len(high_corr))


## 10. Save Final Feature Table

In [ ]:
print(feature_df.shape)
print("Total NaN in weather features:", feature_df[feature_names].isna().sum().sum())  # expect 0

feature_df.to_csv('wildfire_features_full.csv', index=False)
